# CMRI Model — Rebuild Validation & Evidence Figures (SVG + PNG)

**Based on uploaded original sources:** `General_Pretraining_Consolidated_Evidence_v4.ipynb`, `HEART_Chemistry_Slide_Evidence.ipynb`, duplicate evidence notebooks, and cached `validation/evidence/general` figures. This is a **plot-only** notebook: no checkpoint loading, GPU inference, retraining, or modification of original files. 

Produces **editable-text SVG** (`svg.fonttype=none`) and high-resolution PNG. Each figure is saved separately in `summary_figures/cmri_model_replotted/`. Missing numerical sources are reported rather than invented. Geometry uses the **official evaluation summary**; downstream benchmark uses **only full-label (`train_fraction=1`) held-out results**. Original file paths remain unchanged.

Run **Kernel → Restart & Run All** on the HPC `heart_gpu` kernel. Requires `numpy`, `pandas`, `matplotlib`, `scikit-learn`; chemistry coloring needs `rdkit`.

## 0. Set up paths and plotting style

In [ ]:
from pathlib import Path
import json, re, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

PROJECT = Path('/home/ypan/heart_liver/model_train/heart-molecular-pretraining')
GENERAL = Path('/project/yli4/ypan/heart_evaluation/general_audit_10k_v2')
VALIDATION = Path('/home/ypan/heart_liver/model_train/validation')
OUTPUT = PROJECT/'summary_figures'/'cmri_model_replotted'
OUTPUT.mkdir(parents=True,exist_ok=True)
RUN10 = GENERAL/'checkpoint_evaluation'/'full_10k_batch8'
TRAIN10 = GENERAL/'checkpoint_evaluation'/'train_10k_batch8'
BENCH = GENERAL/'chemistry_slide_evidence'/'independent_property_benchmarks'/'benchmark_all_runs.csv'
DUP_CANDIDATES = [Path('/project/yli4/ypan/heart_evaluation/duplicate_impact_expanded_evidence'),
                  Path('/home/ypan/heart_liver/model_train/heart_evaluation/duplicate_impact_expanded_evidence'),
                  Path('/project/yli4/ypan/audits')]
DUPLICATE = next((p for p in DUP_CANDIDATES if (p/'expanded_report.json').is_file()),DUP_CANDIDATES[0])
EVIDENCE = VALIDATION/'evidence'/'general'/'general_pretraining_figures_v4'
PALETTE=['#E9CC79','#DDA65A','#CF8968','#BA706C','#9B6789','#775A82']
plt.rcParams.update({'figure.dpi':120,'savefig.dpi':320,'svg.fonttype':'none',
                     'font.family':'DejaVu Sans','font.size':10,'axes.spines.top':False,
                     'axes.spines.right':False,'axes.titleweight':'semibold'})
GENERATED=[]; SKIPPED=[]; SOURCES={}
def note_skip(name, why):
    SKIPPED.append((name,str(why))); print('SKIPPED:',name,'—',why)
def save(fig,stem,source):
    fig.savefig(OUTPUT/(stem+'.svg'),bbox_inches='tight',facecolor='white')
    fig.savefig(OUTPUT/(stem+'.png'),dpi=320,bbox_inches='tight',facecolor='white')
    plt.show();plt.close(fig)
    GENERATED.append(stem);SOURCES[stem]=source
    print('SAVED:',stem+'.svg and .png')
def read_json(path):
    return json.loads(path.read_text()) if path.is_file() else None
def norm(s):return re.sub('[^a-z0-9]','',str(s).lower())
def col(df,*names):
    mp={norm(k):k for k in df.columns}
    return next((mp[norm(k)] for k in names if norm(k) in mp),None)
def as_pct(v):
    v=float(v);return v*100 if v<=1.000001 else v
VAL=read_json(RUN10/'evaluation_summary.json')
print('H200 10K validation:', 'FOUND' if VAL else 'MISSING', RUN10)
print('Bench CSV:', BENCH.is_file(), BENCH)
print('Duplication audit directory:', DUPLICATE.is_dir())
print('Destination:',OUTPUT)

## 1. H200 validation-only main figure

**Panel a:** validation loss history only; **b:** per-element masked-token recovery; **c:** official aggregate noisy/predicted coordinate RMSE; **d:** validation masked accuracy versus majority-class baseline. No train evaluation statistics are displayed. The optional loss trace reflects a separate training-time validation log, not the held-out 10K experiment.

In [ ]:
history_dir=GENERAL/'training_history'
history=history_dir/'H200_validation_history.csv'
if not history.is_file():
    matches=sorted(history_dir.glob('*validation_history*.csv')) if history_dir.is_dir() else []
    history=matches[0] if matches else history
atoms_path=RUN10/'atom_token_class_accuracy.csv'
if VAL is None or not atoms_path.is_file():
    note_skip('01_validation_main','Missing 10K summary or per-element CSV')
else:
    atom=pd.read_csv(atoms_path)
    symbol=col(atom,'token_symbol','element'); ac=col(atom,'accuracy','masked_atom_accuracy'); count=col(atom,'masked_targets','n','count')
    if not (symbol and ac and count):
        note_skip('01_validation_main',f'Atom columns cannot be mapped: {list(atom.columns)}')
    else:
        fig,axs=plt.subplots(2,2,figsize=(14,9),constrained_layout=True)
        ax=axs[0,0]
        if history.is_file():
            hist=pd.read_csv(history)
            step=col(hist,'num_updates','updates','step'); loss=col(hist,'loss','valid_loss','validation_loss')
            if step and loss:
                xx=pd.to_numeric(hist[step],errors='coerce'); yy=pd.to_numeric(hist[loss],errors='coerce')
                keep=xx.notna()&yy.notna();ax.plot(xx[keep],yy[keep],color=PALETTE[4],lw=1.8)
                ax.set(xlabel='Optimizer updates',ylabel='Logged validation loss',title='a  H200 validation history')
            else:ax.text(.1,.5,'Validation history columns unavailable',transform=ax.transAxes)
        else:ax.text(.1,.5,'Validation history not found',transform=ax.transAxes)
        a=atom.copy();a['_n']=pd.to_numeric(a[count],errors='coerce');a['_acc']=pd.to_numeric(a[ac],errors='coerce')
        a=a[a['_n'].ge(1)&a['_acc'].notna()];a['_acc']=a['_acc'].map(as_pct)
        ax=axs[0,1];xx=np.arange(len(a));bars=ax.bar(xx,a['_acc'],color=[PALETTE[i%len(PALETTE)] for i in range(len(a))]);
        ax.set_xticks(xx,a[symbol].astype(str));ax.set(ylim=(0,106),ylabel='Masked recovery accuracy (%)',title='b  Per-element recovery · validation 10K')
        for bar,countn in zip(bars,a['_n']):ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+1,str(int(countn)),ha='center',fontsize=7)
        ax=axs[1,0];noisy=VAL.get('coord_noisy_rmse');pred=VAL.get('coord_pred_rmse')
        if noisy is not None and pred is not None and float(noisy)>0:
            vals=[float(noisy),float(pred)];bb=ax.bar(['Noisy input','Recovered coordinates'],vals,color=[PALETTE[1],PALETTE[4]],width=.55)
            for bar,v in zip(bb,vals):ax.text(bar.get_x()+bar.get_width()/2,v+.018*max(vals),f'{v:.4f} Å',ha='center',fontsize=9)
            ax.set_ylim(0,max(vals)*1.20)
            ax.text(.5,max(vals)*.83,f'{100*(1-vals[1]/vals[0]):.1f}% lower RMSE',ha='center',fontsize=11)
        else:ax.text(.1,.5,'Official coordinate RMSE unavailable',transform=ax.transAxes)
        ax.set(title='c  Coordinate denoising · official 10K summary',ylabel='Aggregate coordinate RMSE (Å)')
        ax=axs[1,1];acc=VAL.get('masked_atom_accuracy');baseline=VAL.get('masked_atom_majority_class_baseline_accuracy')
        if acc is not None:
            labels=['CMRI model'];vals=[as_pct(acc)]
            if baseline is not None:labels+=['Majority-class baseline'];vals+=[as_pct(baseline)]
            bb=ax.bar(labels,vals,color=[PALETTE[3],PALETTE[0]][:len(vals)],width=.58)
            for bar,v in zip(bb,vals):ax.text(bar.get_x()+bar.get_width()/2,v+1,f'{v:.2f}%',ha='center')
        else:ax.text(.1,.5,'Masked accuracy unavailable',transform=ax.transAxes)
        ax.set(title='d  Overall masked-atom recovery · validation 10K',ylabel='Accuracy (%)',ylim=(0,108))
        fig.suptitle('CMRI Model | H200 general pretraining · validation-only evidence',fontsize=16)
        save(fig,'01_validation_main',{'summary':str(RUN10/'evaluation_summary.json'),'atom':str(atoms_path),'history':str(history)})

## 2. Detailed held-out validation diagnostics

Uses cached molecule/view and batch CSVs. The per-view histogram and coordinate scatter are descriptive diagnostics; their means **are not substituted** for the official summary RMSE. NT-Xent is shown only when stored in the batch file.

In [ ]:
molpath=RUN10/'molecule_diagnostics.csv';batpath=RUN10/'objective_batches.csv'
if not molpath.is_file() or not batpath.is_file():
    note_skip('02_validation_diagnostics','Missing molecule or batch CSV')
else:
    mol=pd.read_csv(molpath);batch=pd.read_csv(batpath)
    fig,axs=plt.subplots(1,3,figsize=(16,4.5),constrained_layout=True)
    ac=col(mol,'masked_atom_accuracy');noisy=col(mol,'coord_noisy_rmse');pred=col(mol,'coord_pred_rmse')
    if ac:
        vals=pd.to_numeric(mol[ac],errors='coerce').dropna();axs[0].hist(vals,bins=30,color=PALETTE[2]);axs[0].set(xlabel='Per-view masked-atom accuracy',ylabel='Count of views')
    else:axs[0].text(.1,.5,'Masked accuracy unavailable',transform=axs[0].transAxes)
    axs[0].set_title('a  Masked accuracy per view')
    if noisy and pred:
        x=pd.to_numeric(mol[noisy],errors='coerce');y=pd.to_numeric(mol[pred],errors='coerce');ok=x.notna()&y.notna()
        axs[1].scatter(x[ok],y[ok],s=6,alpha=.22,color=PALETTE[4]);mx=max(x[ok].max(),y[ok].max()) if ok.any() else 1
        axs[1].plot([0,mx],[0,mx],'--',c='gray',lw=1);axs[1].set(xlabel='Noisy RMSE per view (Å)',ylabel='Predicted RMSE per view (Å)')
    else:axs[1].text(.1,.5,'Coordinate columns unavailable',transform=axs[1].transAxes)
    axs[1].set_title('b  Coordinate recovery per view')
    cl=col(batch,'contrastive_loss','nt_xent_loss');
    if cl:
        z=pd.to_numeric(batch[cl],errors='coerce').dropna();axs[2].hist(z,bins=30,color=PALETTE[1]);axs[2].set(xlabel='Recorded contrastive loss',ylabel='Number of batches')
    else:axs[2].text(.1,.5,'Contrastive loss unavailable',transform=axs[2].transAxes)
    axs[2].set_title('c  Contrastive batch diagnostics')
    fig.suptitle('CMRI Model | Held-out 10K validation diagnostics',fontsize=14)
    save(fig,'02_validation_diagnostics',{'molecules':str(molpath),'batches':str(batpath)})

## 3. Chemistry-colored molecular representation PCA

Uses previously saved 10K embeddings plus the original **aligned sample metadata**. No inference. If the manifest cannot be validated against embedding rows, plotting is skipped rather than falsely assigning descriptor values. PCA color gradients are descriptive and not downstream prediction evidence. **SVG with 10K points can be large.**

In [ ]:
meta_candidates=[GENERAL/'validation_10k_seed17'/'sample_metadata.csv',
                GENERAL/'chemistry_slide_evidence'/'sample_metadata.csv',
                GENERAL/'checkpoint_evaluation'/'full_10k_batch8'/'sample_metadata.csv']
meta_path=next((p for p in meta_candidates if p.is_file()),None)
if meta_path is None and GENERAL.is_dir():
    found=sorted(GENERAL.rglob('sample_metadata.csv'),key=lambda p:len(p.parts));meta_path=found[0] if found else None
cls_path=RUN10/'cls_view0.npy';proj_path=RUN10/'projected_view0.npy'
if not (meta_path and cls_path.is_file() and proj_path.is_file()):
    note_skip('03_chemistry_pca',f'Metadata/CLS/projector inputs missing. Metadata={meta_path}')
else:
    meta=pd.read_csv(meta_path,keep_default_na=False)
    if 'sample_order' in meta:meta=meta.sort_values('sample_order',kind='stable').reset_index(drop=True)
    arrs=[np.load(cls_path,mmap_mode='r'),np.load(proj_path,mmap_mode='r')]
    if not all(a.ndim==2 and a.shape[0]==len(meta) for a in arrs):
        note_skip('03_chemistry_pca',f'Row-count mismatch: manifest={len(meta)}, embeddings={[a.shape for a in arrs]}')
    else:
        if 'heavy_atoms' not in meta or 'cLogP' not in meta:
            if 'smiles' not in meta:
                note_skip('03_chemistry_pca','Manifest has no heavy_atoms/cLogP or smiles')
            else:
                try:
                    from rdkit import Chem
                    from rdkit.Chem import Descriptors
                    mols=[Chem.MolFromSmiles(s) for s in meta.smiles.astype(str)]
                    meta['heavy_atoms']=[m.GetNumHeavyAtoms() if m is not None else np.nan for m in mols]
                    meta['cLogP']=[Descriptors.MolLogP(m) if m is not None else np.nan for m in mols]
                except ImportError:note_skip('03_chemistry_pca','RDKit required for missing molecular descriptors')
        if {'heavy_atoms','cLogP'}<=set(meta.columns):
            from sklearn.decomposition import PCA
            fig,axs=plt.subplots(2,2,figsize=(13,10),constrained_layout=True)
            for i,(label,arr) in enumerate(zip(['CLS','NT-Xent projector'],arrs)):
                pc=PCA(n_components=2,svd_solver='randomized',random_state=17);xy=pc.fit_transform(np.asarray(arr,dtype=np.float32))
                for j,prop in enumerate(['heavy_atoms','cLogP']):
                    vals=pd.to_numeric(meta[prop],errors='coerce').to_numpy();mask=np.isfinite(vals)
                    sc=axs[i,j].scatter(xy[mask,0],xy[mask,1],c=vals[mask],s=3,alpha=.65,cmap='viridis',rasterized=False,linewidths=0)
                    axs[i,j].set(title=f'{label} PCA — {prop}',xlabel=f'PC1 ({pc.explained_variance_ratio_[0]:.1%})',ylabel=f'PC2 ({pc.explained_variance_ratio_[1]:.1%})')
                    fig.colorbar(sc,ax=axs[i,j],label=prop)
            fig.suptitle('CMRI Model | Frozen molecular representations and chemical descriptors',fontsize=15)
            save(fig,'03_chemistry_pca',{'cls':str(cls_path),'projector':str(proj_path),'metadata':str(meta_path)})

## 4. Independent downstream benchmarks — ESOL, FreeSolv, Lipophilicity

**Fixes the missing FreeSolv panel** by reading the real `benchmark` field and normalizing labels. Uses exactly `train_fraction=1.0`, scaffold-held-out test RMSE, averaging across **distinct seeds only**; no mixing with smaller-label fractions. `±SD` means between-seed sample SD. The model is a frozen encoder + Ridge probe, **not RNA delivery prediction**. Do not use historical hardcoded RMSE values.

In [ ]:
def canonical_benchmark(value):
    s=norm(value)
    return {'esol':'ESOL','delaney':'ESOL','freesolv':'FreeSolv','sampl':'FreeSolv','lipophilicity':'Lipophilicity','lipo':'Lipophilicity'}.get(s,None)
def canonical_method(value):
    s=norm(value)
    if 'morgan' in s or 'ecfp' in s:return 'Morgan ECFP4'
    if 'rdkit' in s:return 'RDKit descriptors'
    if 'heavy' in s:return 'Heavy atoms only'
    if 'mean' in s or 'dummy' in s:return 'Mean label'
    if 'frozen' in s or 'heart' in s or 'pretrain' in s or 'cls' in s:return 'Frozen pretrained CLS'
    return str(value).strip()
if not BENCH.is_file():
    note_skip('04_downstream_benchmarks','Benchmark all-runs CSV missing')
else:
    raw=pd.read_csv(BENCH)
    bc=col(raw,'benchmark','dataset','task');mc=col(raw,'model','method');fc=col(raw,'train_fraction','fraction');rc=col(raw,'rmse','test_rmse');sc=col(raw,'seed')
    print('Benchmark columns:',list(raw.columns))
    if not all([bc,mc,fc,rc,sc]):
        note_skip('04_downstream_benchmarks','Missing benchmark/model/train_fraction/rmse/seed columns')
    else:
        d=raw.copy();d['bench']=d[bc].map(canonical_benchmark);d['method']=d[mc].map(canonical_method)
        d['frac']=pd.to_numeric(d[fc],errors='coerce');d['rmse_value']=pd.to_numeric(d[rc],errors='coerce')
        d=d[d['bench'].notna() & np.isclose(d['frac'],1.0,atol=1e-6)&d.rmse_value.notna()].copy()
        dup=d.duplicated(['bench','method',sc],keep=False)
        if dup.any():
            print('WARNING: repeated benchmark/method/seed rows at full fraction. Review these before publication:')
            display(d.loc[dup,[bc,mc,fc,rc,sc]].head(30))
            # Do not silently double weight repeated seeds.
            d=d.drop_duplicates(['bench','method',sc],keep=False)
        agg=d.groupby(['bench','method']).rmse_value.agg(['mean','std','count']).reset_index()
        display(agg.sort_values(['bench','mean']))
        missing=[q for q in ['ESOL','FreeSolv','Lipophilicity'] if q not in set(agg.bench)]
        if missing:note_skip('04_downstream_missing_tasks','Tasks absent after full-label filter: '+', '.join(missing))
        methods=['Frozen pretrained CLS','RDKit descriptors','Morgan ECFP4','Heavy atoms only','Mean label']
        fig,axs=plt.subplots(1,3,figsize=(18,5.8),constrained_layout=True)
        for ax,bench_name in zip(axs,['ESOL','FreeSolv','Lipophilicity']):
            s=agg[agg.bench==bench_name].set_index('method')
            valid=[m for m in methods if m in s.index]
            if not valid:
                ax.text(.5,.5,'No full-label data',transform=ax.transAxes,ha='center');ax.set_title(bench_name);continue
            vals=s.loc[valid,'mean'].to_numpy();sds=s.loc[valid,'std'].fillna(0).to_numpy();ns=s.loc[valid,'count'].astype(int).to_numpy()
            ys=np.arange(len(valid));bars=ax.barh(ys,vals,xerr=sds,capsize=3,color=[PALETTE[3] if m=='Frozen pretrained CLS' else PALETTE[i%len(PALETTE)] for i,m in enumerate(valid)])
            ax.set_yticks(ys,valid,fontsize=9);ax.invert_yaxis();ax.set_title(bench_name);ax.set_xlabel('Scaffold-held-out RMSE (lower better)')
            xmax=max(vals+sds)*1.38 if len(vals) else 1;ax.set_xlim(0,xmax)
            for yy,v,sd,nn in zip(ys,vals,sds,ns):ax.text(v+sd+xmax*.015,yy,f'{v:.3f} ± {sd:.3f} (n={nn})',va='center',fontsize=8)
            ax.grid(axis='x',alpha=.12)
        fig.suptitle('CMRI Model | Frozen CLS + Ridge: independent property benchmarks',fontsize=15)
        save(fig,'04_downstream_benchmarks',{'csv':str(BENCH),'filter':'train_fraction=1; unique seed/method/benchmark'})
        agg.to_csv(OUTPUT/'04_downstream_summary.csv',index=False)

## 5. Labeled-data efficiency (ESOL and Lipophilicity)

Reads the *same* benchmark CSV at multiple train fractions; three seeds are summarized independently for each fraction/method. Does not refit models.

In [ ]:
if BENCH.is_file() and all([bc,mc,fc,rc,sc]):
    full=pd.read_csv(BENCH);full['bench']=full[bc].map(canonical_benchmark);full['method']=full[mc].map(canonical_method)
    full['frac']=pd.to_numeric(full[fc],errors='coerce');full['rmse_value']=pd.to_numeric(full[rc],errors='coerce')
    full=full.dropna(subset=['bench','frac','rmse_value'])
    full=full[full.frac.gt(0)&full.frac.le(1)]
    if full.frac.nunique()<2:note_skip('05_label_efficiency','Only one label fraction in cached CSV')
    else:
        fig,axs=plt.subplots(1,2,figsize=(14,5),constrained_layout=True)
        for ax,bench_name in zip(axs,['ESOL','Lipophilicity']):
            z=full[full.bench==bench_name]
            for i,method in enumerate(['Frozen pretrained CLS','RDKit descriptors','Morgan ECFP4','Heavy atoms only','Mean label']):
                q=z[z.method==method].drop_duplicates(['frac',sc],keep=False)
                if not len(q):continue
                a=q.groupby('frac').rmse_value.agg(['mean','std']).sort_index()
                ax.errorbar(100*a.index,a['mean'],yerr=a['std'].fillna(0),marker='o',capsize=2,label=method,color=PALETTE[i])
            ax.set(title=bench_name,xlabel='Labeled training fraction (%)',ylabel='Held-out RMSE');ax.grid(alpha=.14);ax.legend(fontsize=8)
        fig.suptitle('CMRI Model | Labeled-data efficiency (cached scaffold splits)',fontsize=14)
        save(fig,'05_labeled_data_efficiency',{'csv':str(BENCH)})
else:note_skip('05_label_efficiency','Missing benchmark CSV or required fields')

## 6. Dataset duplication evidence

Replots raw records vs raw-SMILES uniqueness from `expanded_report.json` where accessible. If the report structure does not expose verified totals, no numerical duplicate chart is fabricated. More detailed scaffold, descriptor and 3D analyses are referenced as archived evidence until their exact CSV schema is reviewed.

In [ ]:
rep=DUPLICATE/'expanded_report.json'
data=read_json(rep)
def flatten(d,prefix=''):
    if isinstance(d,dict):
        for k,v in d.items():yield from flatten(v,prefix+'.'+k if prefix else k)
    elif isinstance(d,(int,float)) and not isinstance(d,bool):yield prefix,d
if not data:
    note_skip('06_duplicate_identity','Expanded duplicate report JSON unavailable')
else:
    found=list(flatten(data)); print('Available report numeric keys (first 45):',found[:45])
    def get_num(terms):
        hits=[(k,v) for k,v in found if all(t in norm(k) for t in terms)]
        return hits[0] if len(hits)==1 else None
    total=get_num(['total','record']) or get_num(['raw','record'])
    uniq=get_num(['unique','smiles']) or get_num(['distinct','smiles'])
    if not (total and uniq and float(total[1])>=float(uniq[1])>0):
        note_skip('06_duplicate_identity','Could not unambiguously identify raw/unique counts in JSON; review printed report keys')
    else:
        fig,ax=plt.subplots(figsize=(8,4.5));values=[float(total[1]),float(uniq[1])]
        bars=ax.bar(['Raw LMDB rows','Unique raw SMILES'],values,color=[PALETTE[1],PALETTE[4]],width=.6)
        for bb,v in zip(bars,values):ax.text(bb.get_x()+bb.get_width()/2,v+max(values)*.015,f'{v:,.0f}',ha='center')
        ax.set_ylim(0,max(values)*1.19);ax.set_ylabel('Number of records / identities')
        ax.set_title(f'CMRI Model | Duplicate exposure: {(1-values[1]/values[0])*100:.2f}% excess rows')
        save(fig,'06_duplicate_identity',{'report':str(rep),'raw_key':total[0],'unique_key':uniq[0]})

## 7. Record originals, provenance & export status

Historical figure PNGs are preserved for reference, **not converted into fake editable SVGs**. Only new Matplotlib plots are vector-native. UGI/LNPDB outputs remain separate from general-pretraining validation unless explicitly selected later.

In [ ]:
reference_dirs=[EVIDENCE,PROJECT/'summary_figures'/'recovered_original_figures',
    VALIDATION/'evidence'/'ugi'/'ugi_priority1_interactive_evidence'/'figures',
    VALIDATION/'evidence'/'lnpdb'/'ugi_lnpdb_priority2_evidence'/'figures']
records=[]
for folder in reference_dirs:
    if folder.is_dir():
        for p in sorted(folder.glob('*.png')):records.append({'group':str(folder),'filename':p.name,'source':str(p)})
if records:
    pd.DataFrame(records).to_csv(OUTPUT/'archived_figure_index.csv',index=False)
    print('Indexed',len(records),'existing reference PNGs (no copying/overwriting).')
manifest={'generation_mode':'cached-results-only; no GPU or inference',
          'generated':GENERATED,'skipped':SKIPPED,'sources':SOURCES,
          'note':'SVG text editable; all source outputs preserved'}
(OUTPUT/'provenance_manifest.json').write_text(json.dumps(manifest,indent=2))
print('GENERATED FIGURES:',GENERATED)
print('SKIPPED:',SKIPPED or 'none')
print('OUTPUT DIRECTORY:',OUTPUT)
print('SVG:',len(list(OUTPUT.glob('*.svg'))),'PNG:',len(list(OUTPUT.glob('*.png'))))